In [3]:
import requests
import html
import json
import pandas as pd
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import concurrent.futures
from variables import TOUR_NAME, RK9_BASE_ENDPOINT, RK9_ROSTER

# Rk9 decklist pull

In [4]:
TIMEOUT = 20
NUM_THREADS = 12

In [5]:
with open('archetypes.json', 'r') as file:
    arch_dict = json.load(file)

In [6]:
def rk9_fetch_deck(ref):
    start_target = 'pt-2 px-3 translation lang-EN'.encode('utf-8')
    end_target = '</div>'.encode('utf-8')
    with requests.get(RK9_BASE_ENDPOINT+ref, stream=True, timeout=TIMEOUT) as response:
        try:
            response.raise_for_status()
            decklist_space = False
            deck_string = ""
            for line in response.iter_lines():
                if start_target in line:
                    decklist_space = True
                    continue
                if decklist_space:
                    if end_target in line:
                        break
                    elif b'<li' in line:
                        deck_string += line.decode('utf-8') + '</li>'
        except Exception:
            return {}
    deck_string = html.unescape(deck_string)
    soup = BeautifulSoup(deck_string, 'html.parser')
    li_elements = soup.find_all('li', {'data-cardtype': True})
    decklist = {"pokemon": {}, "trainer": {}, "energy": {}}
    for li in li_elements:
        card_name = li['data-cardname'].split(' - ')[0]
        if card_name in decklist[li['data-cardtype']]:
            decklist[li['data-cardtype']][card_name] += int(li['data-quantity'])
        else:
            decklist[li['data-cardtype']][card_name] = int(li['data-quantity'])
    return decklist

def decklist_distance(decklist, arch_dict):
    if not decklist: return None, 100
    min_d = 12
    deck_archetype = 'Other'
    for deck_name, arch in arch_dict.items():
        d = 0
        for key_card, count in arch['key_cards'].items():
            if key_card in decklist['pokemon']:
                decklist_count = decklist['pokemon'][key_card] 
                card_type = 'pokemon'
            elif key_card in decklist['trainer']:
                decklist_count = decklist['trainer'][key_card] 
                card_type = 'trainer'
            elif key_card in decklist['energy']:
                decklist_count = decklist['energy'][key_card] 
                card_type = 'energy'
            else:
                decklist_count = 0
                card_type = ''
            card_distance = max(0, count - decklist_count)
            d += card_distance if card_type != 'pokemon' else card_distance*4
            if d >= min_d: break
        if d < min_d:
            min_d = d
            deck_archetype = deck_name
    return deck_archetype, min_d

def process_row(tr):
    info_vec = []
    for td in tr.find_all('td'):
        info_vec.append(td.contents[0].strip())
    try:
        # if category and info_vec[4] != category: return {}
        ref = tr.find('a')['href']
        decklist = rk9_fetch_deck(ref)
        archetype, distance = decklist_distance(decklist, arch_dict)
    except Exception:
        print("Error on deck {}{}".format(RK9_BASE_ENDPOINT, ref))
        archetype = None
        distance = 100
    return {
        'Placement': int(info_vec[6]),
        'Player': '{} {}'.format(info_vec[1], info_vec[2]), 
        'Country': info_vec[3], 
        'Category': info_vec[4],
        'Deck': archetype,
        'Ref': ref,
        'Distance': distance
    }

In [15]:
test_pult = rk9_fetch_deck("/decklist/public/EU01wICdQN8zZclF7NTW/7XcvQCYcsKQzwATdJrKE")

In [16]:
decklist_distance(test_pult, arch_dict)

('Dragapult', 3)

In [7]:
response = requests.get(RK9_ROSTER)
soup = BeautifulSoup(response.text, 'html.parser')
table = soup.find('table', {'id': 'dtLiveRoster'})
rows = table.find('tbody').find_all('tr')

In [8]:
results = []
with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    # Submit tasks to the executor
    futures = {executor.submit(process_row, row): row for row in rows}
    
    # Track progress with tqdm
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Rows"):
        row_index = futures[future]
        try:
            result = future.result(timeout=TIMEOUT)
            if result:
                results.append(result)
        except concurrent.futures.TimeoutError:
            print(f"Row {row_index} timed out.")
        except Exception as e:
            print(f"An error occurred on row {row_index}: {e}")


Processing Rows: 100%|██████████| 3960/3960 [16:41<00:00,  3.95it/s]


In [12]:
df = pd.DataFrame(results).sort_values(by='Placement', ascending=True)

In [10]:
decks_masters = df[df['Category'] == 'Masters'][['Placement', 'Player', 'Country', 'Deck']]
decks_senior = df[df['Category'] == 'Senior'][['Placement', 'Player', 'Country', 'Deck']]
decks_junior = df[df['Category'] == 'Junior'][['Placement', 'Player', 'Country', 'Deck']]

In [11]:
with pd.ExcelWriter(f'standings/{TOUR_NAME}_standings.xlsx') as writer:
    # pairings_final_df.to_excel(writer, sheet_name='pairings', index=False)
    decks_masters.to_excel(writer, sheet_name='masters', index=False)
    decks_senior.to_excel(writer, sheet_name='senior', index=False)
    decks_junior.to_excel(writer, sheet_name='junior', index=False)